# ✨ GPT for Text Generation
**Causal Language Modeling with Hugging Face Transformers**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

print(f'PyTorch version    : {torch.__version__}')
print('Libraries loaded ✅')

## 2. Load Dataset
> We use a synthetic dataset of story prompts and continuations. For Causal Language Modeling (CLM), we concatenate the prompt and continuation into a single text sequence.

In [ ]:
df = pd.read_csv('../data/stories.csv')
print(f'Shape   : {df.shape}')

# Combine prompt and continuation into a single text column for CLM
df['text'] = df['prompt'] + ' ' + df['continuation']
print(f'\\nSample text:')
print(df['text'].iloc[0])
df.head()

## 3. Tokenization for Causal LM
> GPT uses Byte-Pair Encoding (BPE). We tokenize the entire text and group them into blocks of a fixed `block_size` (e.g., 64 or 128 tokens). This is different from classification, where we pad to a max length. Here, we pack texts together to maximize training efficiency.

In [ ]:
from datasets import Dataset

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # GPT-2 doesn't have a pad token by default

def tokenize_function(examples):
    # Tokenize and truncate to block_size + 1 (because labels are shifted by 1)
    return tokenizer(examples['text'], truncation=True, max_length=128)

# Create Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[['text']])
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Group texts into blocks (optional but recommended for efficient CLM training)
block_size = 64

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # For CLM, labels are the same as input_ids (the model will shift them internally)
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True)
split_dataset = lm_dataset.train_test_split(test_size=0.1, seed=42)

print(f"Training blocks: {len(split_dataset['train'])}")
print(f"Eval blocks: {len(split_dataset['test'])}")

## 4. Load GPT-2 Model for Causal LM

In [ ]:
# AutoModelForCausalLM is the correct class for GPT-style generation
model = AutoModelForCausalLM.from_pretrained(model_name)

# Resize token embeddings if we added special tokens (not needed here, but good practice)
model.resize_token_embeddings(len(tokenizer))

print(f"Model parameters: {model.num_parameters():,}")
model.config

## 5. Define Training Arguments & Data Collator
> The `DataCollatorForLanguageModeling` handles the dynamic creation of labels by shifting the input sequence by one position, which is the core of next-token prediction.

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False # mlm=False means Causal LM (next token), not Masked LM (like BERT)
)

training_args = TrainingArguments(
    output_dir='./gpt_finetuned',
    learning_rate=2e-4, # Slightly higher LR for small GPT models
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=10,
)

# Perplexity is the standard metric for language modeling (exp(loss))
import math
def compute_metrics(eval_pred):
    loss = eval_pred.loss
    return {'perplexity': math.exp(loss)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized. Ready to train.")

## 6. Train the Model

In [ ]:
# Uncomment to train. For this demo, we'll skip actual training to save time,
# but this is the exact code you would run.

# trainer.train()
# trainer.evaluate()

print("Training code ready. Uncomment trainer.train() to execute.")

## 7. Text Generation (Inference)
> Even without fine-tuning, the base `distilgpt2` model can generate coherent text. Let's test it with the `pipeline` API.

In [ ]:
generator = pipeline("text-generation", model=model_name, tokenizer=tokenizer, framework="pt")

prompts = [
    "The old clock tower struck midnight, and",
    "In the depths of the forgotten forest, a",
    "The artificial intelligence finally gained consciousness, and"
]

print("--- Base Model Generation ---")
for prompt in prompts:
    output = generator(
        prompt, 
        max_length=40, 
        temperature=0.8, 
        top_k=50, 
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f"Prompt: {prompt}")
    print(f"Generated: {output[0]['generated_text']}\n")

## 8. Understanding Causal Language Modeling Loss
> In CLM, the model predicts the next token. The loss is calculated by comparing the model's predictions at position $i$ with the actual token at position $i+1$.

In [ ]:
# Visualize the shift
sample_text = "The cat sat on the mat"
tokens = tokenizer.encode(sample_text, add_special_tokens=False)
decoded_tokens = [tokenizer.decode([t]) for t in tokens]

print(f"Original tokens: {decoded_tokens}")
print(f"Input to model : {decoded_tokens[:-1]}  (predicts next)")
print(f"Target labels  :   {decoded_tokens[1:]}  (what it should predict)")

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
for i, token in enumerate(decoded_tokens[:-1]):
    ax.text(i*1.5, 0.5, token, ha='center', va='center', fontsize=12, 
            bbox=dict(facecolor='#3b82f6', edgecolor='white', boxstyle='round'))
    ax.text(i*1.5 + 0.75, 0.5, '→', ha='center', va='center', fontsize=16, color='#10b981')
    
for i, token in enumerate(decoded_tokens[1:]):
    ax.text(i*1.5 + 1.5, -0.5, token, ha='center', va='center', fontsize=12, 
            bbox=dict(facecolor='#10b981', edgecolor='white', boxstyle='round'))

ax.set_title('Causal LM: Input tokens are shifted to create target labels', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Save Fine-Tuned Model

In [ ]:
# After training, save the model and tokenizer
# model.save_pretrained('../models/fine_tuned_gpt')
# tokenizer.save_pretrained('../models/fine_tuned_gpt')
print("Model saving code ready.")

## 10. Key Takeaways
> - **Causal Language Modeling**: The objective is strictly *next-token prediction*. Labels are created by shifting the input sequence right by one.
> - **Causal Masking**: During training, a look-ahead mask ensures the model cannot "cheat" by attending to future tokens.
> - **Autoregressive Generation**: At inference, the model generates one token, appends it to the input, and repeats until an EOS (End of Sequence) token is generated or max length is reached.
> - **Sampling Matters**: Using `temperature` and `top_k`/`top_p` is crucial for generating diverse, human-like text instead of repetitive loops.